# Week 21 Label Studio Summary

This notebook summarizes the Week 21 ERP-image labeling work: data sources used, labeling volume, pattern-class counts, and the sort variables where at least one pattern instance was found. It writes reproducible CSV tables and PNG plots to `notebooks/week_21/outputs/week21_labeling_summary`.

The actual computation lives in the Julia script `week21_labeling_summary_plots.jl`, mirroring the convention used by the other Week-21 export and CV notebooks (Python notebook drives the Julia script via `subprocess`).

In [ ]:
from pathlib import Path
import csv
import json
import subprocess

from IPython.display import Image, Markdown, display

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / 'notebooks' / 'week_21').exists():
    REPO_ROOT = Path.cwd().parents[1]

WEEK21 = REPO_ROOT / 'notebooks' / 'week_21'
OUTPUT_DIR = WEEK21 / 'outputs' / 'week21_labeling_summary'
TABLES_DIR = OUTPUT_DIR / 'tables'
PLOTS_DIR = OUTPUT_DIR / 'plots'
REPO_ROOT

## 1. Rebuild Tracking Tables and Summary Plots

The first call refreshes the Label Studio annotation tracking CSV from the local SQLite DB. The second call runs the Julia summary script, which reads the refreshed CSV and writes summary tables, plots, and `summary.json` into `outputs/week21_labeling_summary`.

In [ ]:
subprocess.run([
    'python3',
    str(WEEK21 / 'update_labelstudio_annotation_tracking.py'),
], cwd=REPO_ROOT, check=True)

subprocess.run([
    'julia',
    '--project=notebooks/model_test',
    str(WEEK21 / 'week21_labeling_summary_plots.jl'),
], cwd=REPO_ROOT, check=True)

summary = json.loads((OUTPUT_DIR / 'summary.json').read_text())
summary['totals']

## 2. Summary Tables

In [ ]:
def read_csv_rows(path):
    with Path(path).open(newline='') as f:
        return list(csv.DictReader(f))

def markdown_table(rows, columns, max_rows=20):
    shown = rows[:max_rows]
    header = '| ' + ' | '.join(columns) + ' |'
    sep = '| ' + ' | '.join(['---'] * len(columns)) + ' |'
    body = []
    for row in shown:
        values = [str(row.get(col, '')).replace('|', '/') for col in columns]
        body.append('| ' + ' | '.join(values) + ' |')
    if len(rows) > max_rows:
        body.append('| ' + ' | '.join([f'... {len(rows) - max_rows} more rows', *([''] * (len(columns) - 1))]) + ' |')
    return '\n'.join([header, sep, *body])

dataset_summary = read_csv_rows(TABLES_DIR / 'used_data_sources_summary.csv')
export_batch_summary = read_csv_rows(TABLES_DIR / 'export_batch_summary.csv')
positive_sort_variables = read_csv_rows(TABLES_DIR / 'positive_sort_variables_summary.csv')

display(Markdown('### Data sources used'))
display(Markdown(markdown_table(
    dataset_summary,
    ['dataset_key', 'total_labeled', 'pattern_labeled', 'positive_rate', 'n_channels', 'n_sort_variables', 'export_batches', 'excluded_from_training'],
    max_rows=30,
)))

display(Markdown('### Export batches'))
display(Markdown(markdown_table(
    export_batch_summary,
    ['export_batch', 'total_labeled', 'pattern_labeled', 'positive_rate', 'n_datasets'],
    max_rows=20,
)))

In [ ]:
display(Markdown('### Sort variables with at least one pattern-class instance'))
display(Markdown(markdown_table(
    positive_sort_variables,
    ['dataset_key', 'sort_variable', 'export_batches', 'pattern_labeled', 'total_labeled', 'positive_rate', 'pattern_classes', 'channels_with_positive'],
    max_rows=35,
)))

## 3. Plots

In [ ]:
plot_order = [
    ('labeled_images_by_dataset', 'Labeled images by data source'),
    ('positive_rate_by_dataset', 'Pattern-class rate by data source'),
    ('pattern_classes_by_dataset', 'Pattern classes by data source'),
    ('labels_by_export_batch', 'Labeling volume by export batch'),
    ('top_positive_sort_variables', 'Top positive sort variables'),
    ('positive_sort_variable_heatmap', 'Positive sort-variable heatmap'),
    ('annotation_lead_time_by_batch', 'Annotation lead time by export batch'),
]

for key, title in plot_order:
    path = Path(summary['plot_paths'][key])
    display(Markdown(f'### {title}'))
    display(Image(filename=str(path)))

## 4. Output Files

In [ ]:
print('Tables:')
for path in sorted(TABLES_DIR.glob('*.csv')):
    print(' -', path.relative_to(REPO_ROOT))

print('\nPlots:')
for path in sorted(PLOTS_DIR.glob('*.png')):
    print(' -', path.relative_to(REPO_ROOT))